# 🔥 Train YOLOv3-Tiny for fire detection (IoT-AI Fire Extinguisher System)

This notebook trains the **1-class YOLOv3-Tiny** model used by the **AI stage** of the [IoT-AI Fire Extinguisher System](https://github.com/yosrarhim/IoT-AI-Fire-Extinguisher-System), then evaluates it (mAP@0.5) and exports `yolov3-tiny-fire.weights`.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Steps:
1. Check the GPU
2. Get the project and its YOLOv3-Tiny config
3. Build Darknet with CUDA
4. Download pre-trained YOLOv3-Tiny weights (transfer learning)
5. Get a fire dataset (Roboflow **or** your own zip)
6. Prepare the data: keep only the *fire* class, add negatives, split train/valid
7. Train (≈ 1 h on a T4)
8. Evaluate: mAP@0.5, precision, recall, loss chart
9. Test on images
10. Export the weights

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Get the project (YOLOv3-Tiny config lives in models/)
%cd /content
!rm -rf IoT-AI-Fire-Extinguisher-System
!git clone -q https://github.com/yosrarhim/IoT-AI-Fire-Extinguisher-System.git
PROJECT = '/content/IoT-AI-Fire-Extinguisher-System'
!cat {PROJECT}/models/yolov3-tiny-fire.cfg | grep -E 'classes|filters=18|max_batches|steps'

In [ ]:
# 3. Build Darknet (AlexeyAB) with GPU + OpenCV
%cd /content
!rm -rf darknet
!git clone -q https://github.com/AlexeyAB/darknet
%cd /content/darknet
# OpenCV headers are needed for OPENCV=1 (training chart + augmentation)
!apt-get -qq update > /dev/null && apt-get -qq install -y libopencv-dev pkg-config > /dev/null
!sed -i 's/^GPU=0/GPU=1/; s/^CUDNN=0/CUDNN=1/; s/^CUDNN_HALF=0/CUDNN_HALF=1/; s/^OPENCV=0/OPENCV=1/' Makefile
# Recent CUDA versions dropped old GPU architectures (compute_50...): build only for this GPU
import os, subprocess
cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                    capture_output=True, text=True).stdout.split()[0].replace('.', '')
ARCH = f'-gencode arch=compute_{cc},code=[sm_{cc},compute_{cc}]'
print('GPU architecture:', ARCH)
!make -j$(nproc) ARCH="{ARCH}" > build.log 2>&1 ; tail -3 build.log
if not os.path.exists('/content/darknet/darknet'):
    print('Build with OpenCV failed -> rebuilding without OpenCV (training still works, no chart.png)')
    !make clean > /dev/null
    !sed -i 's/^OPENCV=1/OPENCV=0/' Makefile
    !make -j$(nproc) ARCH="{ARCH}" > build.log 2>&1 ; tail -3 build.log
!ls -la darknet

In [ ]:
# 4. Pre-trained YOLOv3-Tiny weights -> first 15 layers (transfer learning)
%cd /content/darknet
!wget -q -O yolov3-tiny.weights https://pjreddie.com/media/files/yolov3-tiny.weights || true
import os
if os.path.exists('yolov3-tiny.weights') and os.path.getsize('yolov3-tiny.weights') > 30_000_000:
    !./darknet partial cfg/yolov3-tiny.cfg yolov3-tiny.weights yolov3-tiny.conv.15 15 > /dev/null
    PRETRAINED = '/content/darknet/yolov3-tiny.conv.15'
    print('Transfer learning from YOLOv3-Tiny (COCO):', PRETRAINED)
else:
    PRETRAINED = ''
    print('Pre-trained weights not available: training from scratch (works, but needs more iterations).')

## 5. Get a fire dataset

Choose **one** option and set `DATASET_SOURCE`.

**Option A – Roboflow (recommended, free):**
1. Create a free account on [roboflow.com](https://roboflow.com) and copy your **API key** (Settings → API Keys).
2. On [Roboflow Universe](https://universe.roboflow.com/search?q=fire%20detection), open a *fire detection* dataset, click **Download Dataset**, choose **YOLO Darknet** → *show download code*, and copy `workspace`, `project` and `version`.

**Option B – your own zip:** upload a zip with images (`.jpg/.png`) and YOLO labels (`.txt`, same name as each image). Add your **ESP32-CAM photos** here: they make the model much better on the real camera.

`FIRE_CLASS_ID` = the class number of *fire* in the dataset labels (check `classes.txt` / `_darknet.labels`; in D-Fire, fire is `1`).

In [ ]:
DATASET_SOURCE = 'roboflow'   # 'roboflow' or 'zip'
FIRE_CLASS_ID = 0             # class id of "fire" in the downloaded labels

# Option A – Roboflow
ROBOFLOW_API_KEY = ''         # paste your key here
RF_WORKSPACE = ''             # e.g. 'some-workspace'
RF_PROJECT = ''               # e.g. 'fire-detection-xxxx'
RF_VERSION = 1

RAW = '/content/raw_dataset'
!rm -rf {RAW} && mkdir -p {RAW}

if DATASET_SOURCE == 'roboflow':
    !pip -q install roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    ds = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download('darknet', location=RAW, overwrite=True)
else:
    from google.colab import files
    up = files.upload()               # choose your dataset .zip
    zip_name = next(iter(up))
    !unzip -q "{zip_name}" -d {RAW}

!find {RAW} -type f | sed 's/.*\.//' | sort | uniq -c

In [ ]:
# 6. Prepare the data: keep only "fire" (remapped to class 0), keep some negatives, split 85/15
import glob, os, random, shutil
random.seed(42)

# Find the "fire" class automatically from the dataset class list (Roboflow: _darknet.labels)
for names_file in glob.glob(f'{RAW}/**/_darknet.labels', recursive=True) + glob.glob(f'{RAW}/**/classes.txt', recursive=True):
    names = [n.strip().lower() for n in open(names_file) if n.strip()]
    if 'fire' in names:
        FIRE_CLASS_ID = names.index('fire')
        print('Classes:', names, '-> fire =', FIRE_CLASS_ID)
        break

DATA = '/content/fire_data'
shutil.rmtree(DATA, ignore_errors=True)
shutil.rmtree('/content/backup', ignore_errors=True)
os.makedirs(f'{DATA}/obj', exist_ok=True)

def label_path(img):
    base = os.path.splitext(img)[0] + '.txt'
    if os.path.exists(base):
        return base
    alt = base.replace(os.sep + 'images' + os.sep, os.sep + 'labels' + os.sep)
    return alt if os.path.exists(alt) else None

images = [p for p in glob.glob(f'{RAW}/**/*', recursive=True)
          if p.lower().endswith(('.jpg', '.jpeg', '.png'))]
positives, negatives = [], []
for i, img in enumerate(images):
    lp = label_path(img)
    boxes = []
    if lp:
        for line in open(lp):
            parts = line.split()
            if len(parts) == 5 and int(float(parts[0])) == FIRE_CLASS_ID:
                boxes.append('0 ' + ' '.join(parts[1:]))
    ext = os.path.splitext(img)[1].lower()
    dst = f'{DATA}/obj/{i:06d}{ext}'
    (positives if boxes else negatives).append((img, dst, boxes))

# negatives (no fire: lamps, red objects...) reduce false alarms; keep up to 30 % of the set
negatives = negatives[: int(len(positives) * 0.43)]
samples = positives + negatives
random.shuffle(samples)
for src, dst, boxes in samples:
    shutil.copy(src, dst)
    open(os.path.splitext(dst)[0] + '.txt', 'w').write('\n'.join(boxes) + ('\n' if boxes else ''))

split = int(len(samples) * 0.85)
open(f'{DATA}/train.txt', 'w').write('\n'.join(d for _, d, _ in samples[:split]) + '\n')
open(f'{DATA}/valid.txt', 'w').write('\n'.join(d for _, d, _ in samples[split:]) + '\n')
open(f'{DATA}/fire.names', 'w').write('fire\n')
os.makedirs('/content/backup', exist_ok=True)
open(f'{DATA}/fire.data', 'w').write(
    f'classes = 1\ntrain = {DATA}/train.txt\nvalid = {DATA}/valid.txt\n'
    f'names = {DATA}/fire.names\nbackup = /content/backup\n')

n_boxes = sum(len(b) for _, _, b in positives)
print(f'{len(positives)} fire images ({n_boxes} boxes) + {len(negatives)} negative images')
print(f'train: {split}   valid: {len(samples) - split}')
assert len(positives) > 0, 'No fire boxes found: check FIRE_CLASS_ID'

In [ ]:
# 7. Train (about 1 hour on a T4). The best model on the validation set is saved as yolov3-tiny-fire_best.weights
%cd /content/darknet
CFG = f'{PROJECT}/models/yolov3-tiny-fire.cfg'
!./darknet detector train {DATA}/fire.data {CFG} {PRETRAINED} -dont_show -map 2>&1 | grep -E "avg loss|mean average precision|Saving weights" | tail -n 40

In [ ]:
# Training curve (loss in blue, mAP in red)
import os
from IPython.display import Image, display
if os.path.exists('/content/darknet/chart.png'):
    display(Image('/content/darknet/chart.png'))
else:
    print('No chart.png (Darknet was built without OpenCV). The mAP in step 8 is what matters.')

In [ ]:
# 8. Evaluate the best model: precision, recall, F1, mAP@0.5
%cd /content/darknet
BEST = '/content/backup/yolov3-tiny-fire_best.weights'
!./darknet detector map {DATA}/fire.data {CFG} {BEST} -iou_thresh 0.5 2>&1 | grep -E "precision|mean average precision|F1|TP ="

In [ ]:
# 9. Test on validation images with the same code as the project (OpenCV DNN + yolo_fire.py)
import sys, cv2, random
import matplotlib.pyplot as plt
sys.path.insert(0, PROJECT)
from yolo_fire import YoloV3TinyFireDetector

det = YoloV3TinyFireDetector(CFG, BEST, conf_threshold=0.4)
valid = open(f'{DATA}/valid.txt').read().split()
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, path in zip(axes.flat, random.sample(valid, min(6, len(valid)))):
    img = cv2.imread(path)
    dets = det.detect(img)
    det.draw(img, dets)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(f'{len(dets)} fire box(es)'); ax.axis('off')
plt.tight_layout()
plt.savefig('/content/yolo_fire_results.png', dpi=110)
plt.show()

In [ ]:
# 10. Export: download the weights, the results image and the training chart
import shutil
shutil.copy(BEST, '/content/yolov3-tiny-fire.weights')
import os
outputs = ['/content/yolov3-tiny-fire.weights', '/content/yolo_fire_results.png']
if os.path.exists('/content/darknet/chart.png'):
    shutil.copy('/content/darknet/chart.png', '/content/yolo_training_chart.png')
    outputs.append('/content/yolo_training_chart.png')
from google.colab import files
for f in outputs:
    files.download(f)

## Next steps

1. Put `yolov3-tiny-fire.weights` in the `models/` folder of the project.
2. Add `yolo_fire_results.png` and `yolo_training_chart.png` to `docs/` and write the mAP@0.5 in the README.
3. Run the full system:
```bash
python flame_detector.py --stream http://<ESP32-CAM-IP>:81/stream --broker <BROKER-IP> \
    --yolo-weights models/yolov3-tiny-fire.weights
```